In [5]:
import pandas as pd

PATH = "/kaggle/input/datasets/dinger4/club-signups/club_signups.csv"

# dtype=str: read everything as text so pandas doesn't guess (e.g. turn "1" into a number)
raw = pd.read_csv(PATH, dtype=str)
df = raw.copy()          # raw is never touched, so we can compare against it later

print("Shape:", df.shape)
print(df.dtypes, "\n")
for col in ["faculty", "club", "city", "fee_paid"]:
    print(df[col].value_counts(dropna=False), "\n")

Shape: (39, 9)
signup_id       object
student_id      object
full_name       object
email           object
faculty         object
city            object
club            object
signed_up_at    object
fee_paid        object
dtype: object 

faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
iet                                       3
MET                                       3
Engineering and Materials Science         1
Pharma                                    1
Media Engineering and Technology          1
met                                       1
pharmacy                                  1
ems                                       1
Name: count, dtype: int64 

club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Football       3
Chess Club     3
Soccer         2
music        

In [6]:
def normalise(s):
    # strip ends, lower-case, collapse repeated inner spaces
    return s.str.strip().str.lower().str.replace(r"\s+", " ", regex=True)

faculty_map = {
    "met": "MET", "media engineering and technology": "MET",
    "iet": "IET", "information engineering and technology": "IET",
    "ems": "EMS", "engineering and materials science": "EMS",
    "pharmacy": "Pharmacy", "pharma": "Pharmacy",
    "management": "Management",
}
club_map = {
    "robotics": "Robotics",
    "debate": "Debate", "debating": "Debate", "debate club": "Debate",
    "music": "Music",
    "football": "Football", "soccer": "Football",
    "chess": "Chess", "chess club": "Chess",
}
city_map = {
    "cairo": "Cairo",
    "giza": "Giza", "el giza": "Giza",
    "alexandria": "Alexandria", "alex": "Alexandria",
}

for col, mapping in [("faculty", faculty_map), ("club", club_map), ("city", city_map)]:
    df[col] = normalise(df[col]).map(mapping)
    assert df[col].notna().all(), f"unmapped values in {col}"   # .map gives NaN for anything missing from the dict

assert set(df["faculty"]) <= {"MET", "IET", "EMS", "Pharmacy", "Management"}
assert set(df["club"])    <= {"Robotics", "Debate", "Music", "Football", "Chess"}
assert set(df["city"])    <= {"Cairo", "Giza", "Alexandria"}
print("faculty, club, city are canonical")

faculty, club, city are canonical


In [7]:
# Names: trim, collapse inner spaces, same capitalisation for everyone (Title Case)
df["full_name"] = (df["full_name"].str.strip()
                                  .str.replace(r"\s+", " ", regex=True)
                                  .str.title())

# Emails: trim + lower-case
df["email"] = df["email"].str.strip().str.lower()

# fee_paid -> real boolean
fee_map = {"yes": True, "y": True, "true": True, "1": True,
           "no": False, "n": False, "false": False, "0": False}
df["fee_paid"] = df["fee_paid"].str.strip().str.lower().map(fee_map)
assert df["fee_paid"].notna().all()
df["fee_paid"] = df["fee_paid"].astype(bool)
print(df["fee_paid"].dtype)

bool


In [8]:
s = df["signed_up_at"].str.strip()

is_iso = s.str.match(r"^\d{4}-\d{2}-\d{2} ")     # format 1: 2026-09-15 13:39
print("ISO format rows:", is_iso.sum(), "| slash format rows:", (~is_iso).sum())

# Format 2 is 15/09/2026 03:12. Values like 15/09 and 16/09 prove the first number is the DAY
# (there is no month 15), so the format is day/month/year.
print(s[~is_iso].str.split("/").str[0].astype(int).max(), "-> first part goes above 12, so it must be the day")

parsed = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
parsed[is_iso]  = pd.to_datetime(s[is_iso],  format="%Y-%m-%d %H:%M")
parsed[~is_iso] = pd.to_datetime(s[~is_iso], format="%d/%m/%Y %H:%M")
df["signed_up_at"] = parsed

assert df["signed_up_at"].notna().all()
print(df["signed_up_at"].min(), "->", df["signed_up_at"].max())   # all within 13-18 Sep 2026: confirms day/month are right

# What if we de-duplicated student_id + club on the RAW data (spellings not fixed yet)?
raw_no_exact = raw.drop_duplicates()
print("Exact duplicates in raw:", raw.duplicated().sum())
print("student+club duplicates found on RAW (after exact removal):",
      raw_no_exact.duplicated(["student_id", "club"]).sum())

# Same check on the cleaned data (exact copies removed the same way)
clean_no_exact = df.drop_duplicates()
print("student+club duplicates found on CLEANED:",
      clean_no_exact.duplicated(["student_id", "club"]).sum())

ISO format rows: 25 | slash format rows: 14
18 -> first part goes above 12, so it must be the day
2026-09-13 14:36:00 -> 2026-09-18 23:36:00
Exact duplicates in raw: 3
student+club duplicates found on RAW (after exact removal): 2
student+club duplicates found on CLEANED: 4


In [9]:
n_start = len(raw)
n_before = len(df)

# B.1 exact duplicates (every column identical, including signup_id)
n_exact = df.duplicated().sum()
print("Exact duplicates:", n_exact)
df = df.drop_duplicates()
n_after_exact = len(df)

# B.2 same student + same club, different signup_id: keep the LATEST submission
df = (df.sort_values(["signed_up_at", "signup_id"])          # oldest -> newest (signup_id only breaks ties)
        .drop_duplicates(["student_id", "club"], keep="last"))
n_resubmits = n_after_exact - len(df)
print("Re-submitted sign-ups removed:", n_resubmits)

df = df.sort_values("signup_id").reset_index(drop=True)

# B.3 check
assert not df.duplicated(["student_id", "club"]).any()
print(f"{n_start} rows -> -{n_exact} exact -> -{n_resubmits} resubmitted -> {len(df)} left")

Exact duplicates: 3
Re-submitted sign-ups removed: 4
39 rows -> -3 exact -> -4 resubmitted -> 32 left


In [10]:
print(df.dtypes)
display(df)

df.to_csv("club_signups_clean.csv", index=False)

signup_id               object
student_id              object
full_name               object
email                   object
faculty                 object
city                    object
club                    object
signed_up_at    datetime64[ns]
fee_paid                  bool
dtype: object


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S001,58-9118,Mariam Kamel,mariam.kamel@student.guc.edu.eg,IET,Giza,Robotics,2026-09-13 14:36:00,True
1,S002,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-13 15:38:00,True
2,S003,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Chess,2026-09-13 21:16:00,True
3,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess,2026-09-13 22:36:00,True
4,S005,61-4733,Omar Adel,omar.adel@student.guc.edu.eg,EMS,Cairo,Football,2026-09-13 23:52:00,True
5,S006,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Football,2026-09-14 01:22:00,True
6,S007,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Robotics,2026-09-14 17:03:00,True
7,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32:00,True
8,S009,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Robotics,2026-09-15 01:05:00,True
9,S010,61-3846,Ahmed Zaki,ahmed.zaki@student.guc.edu.eg,Management,Alexandria,Chess,2026-09-15 03:12:00,False
